# From step 4, we determined the top 5 molecular properties that contribute to a higher interactivity between molecule and TRPV1 receptor.

Why? When we make future predictions, we will now also be have additional predicted attributes that we can use to find trends in how they affect the pChEMBL values. Of course, the predicted values are based on pattern recongition.

Let's now train another chemprop model with these properties:
https://chemprop.readthedocs.io/en/latest/multi_task.html

**Here is my current thought process as of 2/19/26**

We need three components to identify a viable novel non-opoid TRPV1 agonist(s):
- A model to predict values based on structure (such as this chemprop model)
- An intuitive compound generator that takes in molecular properties (such as ChemTSV)
- A positive feedback loop that encourages the compound generator to make a more "potent compound".

In [ ]:
# Install ChemProp - install Pytorch if you haven't already, Google Colab already has PyTorch installed.
!pip install chemprop

In [ ]:
from lightning import pytorch as pl
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from chemprop import data, models, nn

In [ ]:
input_path = '/content/drive/MyDrive/Colab_Notebooks/TRPV1-drug-discovery-research/train_test_data/trpv1_agonist_data_trends.csv'
df_input = pd.read_csv(input_path)
df_input.head(5)


,molecule_chembl_id,canonical_smiles,pchembl_value,alogp,aromatic_rings,full_mwt,hba,hbd,heavy_atoms,mw_freebase,np_likeness_score,num_ro5_violations,psa,qed_weighted,rtb
0,CHEMBL514691,Cc1nc2cc(NC(=O)c3ccc(-c4ccc(F)cc4)nc3C)ccc2s1,7.20,5.37,4.0,377.44,4.0,1.0,27.0,377.44,-2.26,1.0,54.88,0.52,3.0
1,CHEMBL514691,Cc1nc2cc(NC(=O)c3ccc(-c4ccc(F)cc4)nc3C)ccc2s1,7.31,5.37,4.0,377.44,4.0,1.0,27.0,377.44,-2.26,1.0,54.88,0.52,3.0
2,CHEMBL514691,Cc1nc2cc(NC(=O)c3ccc(-c4ccc(F)cc4)nc3C)ccc2s1,8.05,5.37,4.0,377.44,4.0,1.0,27.0,377.44,-2.26,1.0,54.88,0.52,3.0
3,CHEMBL514691,Cc1nc2cc(NC(=O)c3ccc(-c4ccc(F)cc4)nc3C)ccc2s1,7.40,5.37,4.0,377.44,4.0,1.0,27.0,377.44,-2.26,1.0,54.88,0.52,3.0
4,CHEMBL514691,Cc1nc2cc(NC(=O)c3ccc(-c4ccc(F)cc4)nc3C)ccc2s1,7.00,5.37,4.0,377.44,4.0,1.0,27.0,377.44,-2.26,1.0,54.88,0.52,3.0


In [ ]:

smiles_column = 'canonical_smiles'
target_columns = ["pchembl_value", "alogp", "hba", "psa", "rtb", "np_likeness_score"]

smis = df_input.loc[:, smiles_column].values
ys = df_input.loc[:, target_columns].values

datapoints = [data.MoleculeDatapoint.from_smi(smi, y) for smi, y in zip(smis, ys)]

In [ ]:
split_indices = data.make_split_indices(datapoints, seed=42)
train_data, val_data, test_data = data.split_data_by_indices(datapoints, *split_indices)

train_dset = data.MoleculeDataset(train_data[0])
val_dset = data.MoleculeDataset(val_data[0])
test_dset = data.MoleculeDataset(test_data[0])

In [ ]:
output_scaler = train_dset.normalize_targets()
val_dset.normalize_targets(output_scaler)

train_loader = data.build_dataloader(train_dset)
val_loader = data.build_dataloader(val_dset, shuffle=False)
test_loader = data.build_dataloader(test_dset, shuffle=False)

In [ ]:
output_transform = nn.transforms.UnscaleTransform.from_standard_scaler(output_scaler)

ffn = nn.RegressionFFN(n_tasks = len(target_columns), output_transform=output_transform)

metric_list = [nn.metrics.RMSE(), nn.metrics.MAE()]
chemprop_model = models.MPNN(nn.BondMessagePassing(), nn.MeanAggregation(), ffn, metric_list)

In [ ]:
from lightning.pytorch.callbacks import ModelCheckpoint

checkpointing = ModelCheckpoint(
    "checkpoints",  # Directory where model checkpoints will be saved
    "best-{epoch}-{val_loss:.2f}",  # Filename format for checkpoints, including epoch and validation loss
    "val_loss",  # Metric used to select the best checkpoint (based on validation loss)
    mode="min",  # Save the checkpoint with the lowest validation loss (minimization objective)
    save_last=True,  # Always save the most recent checkpoint, even if it's not the best
)

trainer = pl.Trainer(logger=False, enable_checkpointing=True, max_epochs=60, callbacks=[checkpointing])

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
trainer.fit(chemprop_model, train_loader, val_loader)

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loading `train_dataloader` to estimate number of stepping batches.
INFO:lightning.pytorch.utilities.rank_zero:Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation    │      0 │ train │     0 │
│ 2 │ bn              │ Identity           │      0 │ train │     0 │
│ 3 │ predictor       │ RegressionFFN      │ 92.1 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList         │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 319 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 319 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 24                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:485: Your 
`val_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for 
val/test dataloaders.

INFO: `Trainer.fit` stopped: `max_epochs=60` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=60` reached.


In [ ]:
# Get validating/testing MAE and RMSE
val = trainer.validate(chemprop_model, val_loader)

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          val/mse          │     0.097699373960495     │
│         val_loss          │    0.0976993590593338     │
└───────────────────────────┴───────────────────────────┘

In [ ]:
# Get testing MAE and RMSE
trainer.test(dataloaders=test_loader, weights_only=False)

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/checkpoint_connector.py:149: `.test(ckpt_path=None)` was called without a model. The best model of the previous `fit` call will be used. You can pass `.test(ckpt_path='best')` to use the best model or `.test(ckpt_path='last')` to use the last model. If you pass a value, this warning will be silenced.
INFO: Restoring states from the checkpoint path at /content/checkpoints/best-epoch=59-val_loss=0.10.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at /content/checkpoints/best-epoch=59-val_loss=0.10.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loaded model weights from the checkpoint at /content/checkpoints/best-epoch=59-val_loss=0.10.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Loaded model weights from the checkpoint at /content/checkpoints/best-epoch=59-val_loss=0.10.ckp

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:485: Your `test_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/mse          │     2.804313898086548     │
└───────────────────────────┴───────────────────────────┘

[{'test/mse': 2.804313898086548}]

In [ ]:
# End script here
assert False

AssertionError: 

In [ ]:
best_model = !find /content/checkpoints/ -name 'best*'
best_model[0]

In [ ]:
# Run this cell to downlaod the best model file
from google.colab import files

# Download the .ckpt model file
files.download(best_model[0])